# 10 · Ceiling analysis: how much MCC is left above a single tree, and at what cost on the ESP32?

On NF-ToN-IoT-v2 the selected tree reaches MCC ≈ 0.972 and an unconstrained tree with about 616,000 nodes reaches 0.9727. The remaining errors are mostly
confusions between similar attack families (xss ↔ injection ↔ password, dos ↔ ddos). This notebook measures whether larger models or derived features would help.

Same data, feature policy and split, and the same training rows for every model:

| Family | Models | Question |
|---|---|---|
| Single tree | saved configuration; + derived features | do cheap ratio features help? |
| Unconstrained tree | default tree, no limits | single-tree ceiling |
| Random forest / extra trees | several sizes, regularised like the tree | does averaging help? |
| Boosted trees | XGBoost / LightGBM, several sizes | strong-model ceiling |
| Forest + derived features | largest forest | features and ensembles combined |

For every model the notebook reports validation MCC and macro-F1, the estimated size on the ESP32 (trees: 11 B per node; boosted trees: 14 B per node) and whether
it fits the flash budget. **Selection uses validation only**; the test split is scored once, for three models (baseline, best within budget, best overall).

In [ ]:
# ---- Setup: find the package root, work inside results/, import the shared code ----
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "framework.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
os.environ["PAPER_PACKAGE_ROOT"] = str(ROOT)

# >>> WHICH DATASET?  nf_ton_iot_v2 | cic_ton_iot | nf_unsw_nb15_v3
# run_all.py sets ACTIVE_DATASET; when running by hand, edit the default below
os.environ.setdefault("ACTIVE_DATASET", "nf_unsw_nb15_v3")

sys.path.insert(0, str(ROOT / "src"))
(ROOT / "results").mkdir(exist_ok=True)
os.chdir(ROOT / "results")            # every output (csv / png / pkl / headers) lands under results/

from run_settings import *            # PAPER_MODE: full | quick | smoke

In [ ]:
from eval_helpers import *
from framework import _json_default
import derived_features as dfx
import time, datetime
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
print("Dataset:", ACTIVE_DATASET, "| train rows per model:", f"{HR_TRAIN_ROWS:,}", "| repeats:", HR_REPEATS, "| forests:", HR_FOREST_SIZES, "| boosting rounds:", HR_BOOST_ROUNDS)
print("XGBoost available:", HAS_XGB, "| LightGBM available:", HAS_LGBM, "| ESP32 model budget:", HR_MODEL_BUDGET_KB, "KB")

## 1 · Data, split, baseline configuration

In [ ]:
X, y, le, numeric_cols = load_publishability_data()
class_names = list(le.classes_)
X_train, X_val, X_test, y_train, y_val, y_test, _, _ = same_split(X, y)
rare_classes, extremely_rare_classes = infer_rare_classes(y_train)
Xv, yv = stratified_sample(X_val, y_val, HR_EVAL_ROWS, SEED)
Xt, yt = stratified_sample(X_test, y_test, HR_EVAL_ROWS, SEED)

try:
    artifact, metadata = load_artifact_and_metadata()
    BASE_PARAMS, BASE_K = dict(metadata["best_params"]), int(metadata["best_k"])
    print("baseline = saved model:", Path(metadata["model_path"]).name)
except FileNotFoundError:
    BASE_PARAMS, BASE_K = {"max_depth": 20, "min_samples_leaf": 8}, 30
    print("no saved model found -> baseline = max_depth 20, min_samples_leaf 8")
BASE_K = min(BASE_K, len(numeric_cols))
print("baseline parameters:", BASE_PARAMS, "| K =", BASE_K)

Xv_d, new_cols = dfx.add_derived_features(Xv); Xt_d, _ = dfx.add_derived_features(Xt)
print(f"{len(numeric_cols)} base features; derived features added: {len(new_cols)} -> {new_cols}")
print(f"validation rows {len(Xv):,} | test rows {len(Xt):,}  (test used once, at the end)")

## 2 · The model ladder

In [ ]:
def tree_kw():
    kw = {k: v for k, v in BASE_PARAMS.items() if k in ("criterion", "max_depth", "min_samples_leaf", "min_samples_split", "ccp_alpha", "class_weight")}
    return kw

def make_models(n_train):
    ms = {}
    ms["Single tree (current config)"] = ("tree", "base", lambda: DecisionTreeClassifier(random_state=SEED, **BASE_PARAMS))
    ms["Single tree + derived features"] = ("tree", "derived", lambda: DecisionTreeClassifier(random_state=SEED, **BASE_PARAMS))
    ms["Unconstrained tree (default)"] = ("tree", "base", lambda: DecisionTreeClassifier(random_state=SEED))
    ms_frac = min(1.0, 1_000_000 / max(n_train, 1))                 # each tree sees at most ~1M bootstrap rows
    for n in HR_FOREST_SIZES:
        kw = {k: v for k, v in tree_kw().items() if k != "class_weight"}
        ms[f"RandomForest x{n}"] = ("forest", "base", lambda n=n, kw=kw: RandomForestClassifier(n_estimators=n, bootstrap=True, max_samples=ms_frac if ms_frac < 1 else None, n_jobs=-1, random_state=SEED, **kw))
        ms[f"ExtraTrees x{n}"] = ("forest", "base", lambda n=n, kw=kw: ExtraTreesClassifier(n_estimators=n, bootstrap=True, max_samples=ms_frac if ms_frac < 1 else None, n_jobs=-1, random_state=SEED, **kw))
    big = max(HR_FOREST_SIZES); kw = {k: v for k, v in tree_kw().items() if k != "class_weight"}
    ms[f"RandomForest x{big} + derived"] = ("forest", "derived", lambda: RandomForestClassifier(n_estimators=big, bootstrap=True, max_samples=ms_frac if ms_frac < 1 else None, n_jobs=-1, random_state=SEED, **kw))
    for r in HR_BOOST_ROUNDS:
        if HAS_XGB:
            ms[f"XGBoost {r} rounds"] = ("boosted", "base", lambda r=r: XGBClassifier(n_estimators=r, max_depth=6, learning_rate=0.3, subsample=0.8, tree_method="hist", n_jobs=-1, random_state=SEED, eval_metric="mlogloss"))
        if HAS_LGBM:
            ms[f"LightGBM {r} rounds"] = ("boosted", "base", lambda r=r: LGBMClassifier(n_estimators=r, num_leaves=63, learning_rate=0.2, subsample=0.8, subsample_freq=1, n_jobs=-1, random_state=SEED, verbose=-1))
    return ms

def model_stats(clf, family):
    """(total nodes, trees, max depth, estimated exported KB).  Trees: 11 B/node (3x int16 + float32 + uint8); boosted trees: 14 B/node (float32 leaf value)."""
    if family == "tree":
        return int(clf.tree_.node_count), 1, int(clf.get_depth()), 11 * clf.tree_.node_count / 1024
    if family == "forest":
        nodes = sum(int(e.tree_.node_count) for e in clf.estimators_); return nodes, len(clf.estimators_), max(int(e.get_depth()) for e in clf.estimators_), 11 * nodes / 1024
    if hasattr(clf, "get_booster"):
        tdf = clf.get_booster().trees_to_dataframe(); nodes = int(len(tdf)); return nodes, int(tdf["Tree"].nunique()), clf.get_params().get("max_depth") or 0, 14 * nodes / 1024
    tdf = clf.booster_.trees_to_dataframe(); nodes = int(len(tdf)); return nodes, int(tdf["tree_index"].nunique()), 0, 14 * nodes / 1024

print(list(make_models(HR_TRAIN_ROWS)))

## 3 · Train and score on the validation split  *(resumable results are not needed: each repeat is a separate training sample)*

In [ ]:
OUT = Path("headroom") / ACTIVE_DATASET; OUT.mkdir(parents=True, exist_ok=True)
rows, kept = [], {}
t_start = time.perf_counter()
for rep in range(HR_REPEATS):
    seed = SEED + 1300 + rep
    Xtr, ytr = stratified_sample(X_train, y_train, HR_TRAIN_ROWS, seed)
    Xtr_d, _ = dfx.add_derived_features(Xtr)
    for name, (family, feats, factory) in make_models(len(Xtr)).items():
        cols_tr, cols_v = (Xtr_d, Xv_d) if feats == "derived" else (Xtr, Xv)
        k = min(BASE_K, cols_tr.shape[1])
        t0 = time.perf_counter()
        pipe = build_safe_pipeline(factory(), k=None if name.startswith("Unconstrained") else k); pipe.fit(cols_tr, ytr); fit_s = time.perf_counter() - t0
        pv = pipe.predict(cols_v); mv = compute_enhanced_metrics(yv, pv, rare_classes, extremely_rare_classes)
        nodes, trees, depth, kb = model_stats(pipe.named_steps["clf"], family)
        rows.append({"model": name, "family": family, "features": feats, "repeat": rep, "train_rows": len(Xtr), "val_mcc": mv["mcc"], "val_macro_f1": mv["macro_f1"],
                     "val_accuracy": mv["accuracy"], "val_minority_recall": mv["minority_recall"], "total_nodes": nodes, "trees": trees, "max_depth": depth,
                     "exported_kb_est": kb, "fits_budget": kb <= HR_MODEL_BUDGET_KB, "fit_seconds": fit_s})
        if rep == 0: kept[name] = (pipe, feats)                     # keep repeat-0 models for the test scoring and error analysis
        else: del pipe
        print(f"rep {rep+1}/{HR_REPEATS} | {name:34s} val MCC {mv['mcc']:.4f} macro-F1 {mv['macro_f1']:.4f} | nodes {nodes:>9,} | ~{kb:8.0f} KB | {fit_s:5.0f}s", flush=True)
raw = pd.DataFrame(rows); raw.to_csv(OUT / "headroom_raw.csv", index=False)
print(f"\ntotal {time.perf_counter() - t_start:.0f}s")

## 4 · Results (validation)

In [ ]:
raw = pd.read_csv(OUT / "headroom_raw.csv")
g = raw.groupby("model", sort=False)
summ = pd.DataFrame({"family": g["family"].first(), "features": g["features"].first(), "val_mcc": g["val_mcc"].mean(), "val_mcc_std": g["val_mcc"].std(),
                     "val_macro_f1": g["val_macro_f1"].mean(), "val_minority_recall": g["val_minority_recall"].mean(), "total_nodes": g["total_nodes"].mean(), "trees": g["trees"].first(),
                     "exported_kb_est": g["exported_kb_est"].mean(), "fits_budget": g["fits_budget"].first(), "fit_seconds": g["fit_seconds"].mean()})
BASE = "Single tree (current config)"
summ["gain_vs_baseline_mcc"] = summ["val_mcc"] - summ.loc[BASE, "val_mcc"]
summ = summ.sort_values("val_mcc", ascending=False); summ.to_csv(OUT / "headroom_summary.csv"); display(summ.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), constrained_layout=True)
famc = {"tree": "#1f77b4", "forest": "#2ca02c", "boosted": "#d95f02"}
ax = axes[0]
for nme, r in summ.iterrows():
    ax.errorbar(r["exported_kb_est"], r["val_mcc"], yerr=0 if np.isnan(r["val_mcc_std"]) else r["val_mcc_std"], fmt="o" if r["features"] == "base" else "s", color=famc[r["family"]], capsize=2)
    ax.annotate(nme.replace("RandomForest", "RF").replace("ExtraTrees", "ET").replace("Single tree", "tree"), (r["exported_kb_est"], r["val_mcc"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
ax.axvline(HR_MODEL_BUDGET_KB, color="red", ls="--", lw=1); ax.text(HR_MODEL_BUDGET_KB, ax.get_ylim()[0], " flash budget", color="red", fontsize=7, va="bottom")
ax.set_xscale("log"); ax.set_xlabel("estimated exported model size (KB, log)"); ax.set_ylabel("validation MCC")
ax.set_title("accuracy vs ESP32 cost  (square = with derived features)")
ax = axes[1]; s2 = summ.sort_values("gain_vs_baseline_mcc")
ax.barh(s2.index, s2["gain_vs_baseline_mcc"], color=[famc[f] for f in s2["family"]])
for i, (nme, r) in enumerate(s2.iterrows()):
    if not r["fits_budget"]: ax.text(0, i, "  exceeds flash budget", fontsize=6, va="center", color="gray")
ax.axvline(0, color="black", lw=.8); ax.axvline(.005, color="green", ls=":", lw=1); ax.set_xlabel("validation MCC gain over the single tree"); ax.tick_params(axis="y", labelsize=7)
ax.set_title("headroom above the single tree (green dotted = 0.005, 'worth building')")
fig.suptitle(f"{ACTIVE_DATASET}: how much MCC is left above a single decision tree?")
fig.savefig(FIGURES_DIR / "headroom_analysis.png", dpi=300, bbox_inches="tight"); plt.show()

## 5 · Where do the errors come from?  *(baseline tree vs the best model that fits the flash budget)*

In [ ]:
fit_ok = summ[summ.fits_budget]; best_fit = fit_ok.index[0]; best_all = summ.index[0]
print(f"best model that fits the budget: {best_fit}  (val MCC {summ.loc[best_fit, 'val_mcc']:.4f})  |  best overall: {best_all}  (val MCC {summ.loc[best_all, 'val_mcc']:.4f})")

def val_pred(name):
    pipe, feats = kept[name]; return pipe.predict(Xv_d if feats == "derived" else Xv)

def confusion_pairs(pred, top=8):
    cmx = confusion_matrix(yv, pred, labels=np.arange(len(class_names))); err = cmx.sum() - np.trace(cmx)
    pairs = [(class_names[i], class_names[j], int(cmx[i, j])) for i in range(len(class_names)) for j in range(len(class_names)) if i != j and cmx[i, j] > 0]
    pairs.sort(key=lambda t: -t[2]); return pd.DataFrame(pairs[:top], columns=["true", "predicted", "count"]).assign(share_of_errors=lambda d: (d["count"] / max(err, 1)).round(3)), int(err)

for nm in dict.fromkeys([BASE, best_fit, best_all]):
    pr = val_pred(nm); tbl, err = confusion_pairs(pr); print(f"\n--- {nm}: {err:,} errors ({err/len(yv):.2%} of validation rows) ---"); display(tbl)

rec = {}
for nm in dict.fromkeys([BASE, best_fit, best_all]):
    pr = val_pred(nm); _, rc, _, sp = precision_recall_fscore_support(yv, pr, labels=np.arange(len(class_names)), zero_division=0); rec[nm] = rc
rec_df = pd.DataFrame(rec, index=class_names); rec_df["support"] = sp; rec_df.to_csv(OUT / "headroom_per_class_recall.csv"); display(rec_df.round(3))

## 6 · Test score — once, for three models only  *(never used for selection)*

In [ ]:
trows = []
for nm in dict.fromkeys([BASE, best_fit, best_all]):
    pipe, feats = kept[nm]; pt = pipe.predict(Xt_d if feats == "derived" else Xt); mt = compute_enhanced_metrics(yt, pt, rare_classes, extremely_rare_classes)
    trows.append({"model": nm, "role": "baseline" if nm == BASE else ("best that fits the budget" if nm == best_fit else "") + (" / best overall" if nm == best_all else ""),
                  "test_mcc": mt["mcc"], "test_macro_f1": mt["macro_f1"], "test_accuracy": mt["accuracy"], "exported_kb_est": summ.loc[nm, "exported_kb_est"], "total_nodes": summ.loc[nm, "total_nodes"]})
test_df = pd.DataFrame(trows); test_df.to_csv(OUT / "headroom_test.csv", index=False); display(test_df.round(4))

## 7 · Summary

In [ ]:
gain_fit = summ.loc[best_fit, "val_mcc"] - summ.loc[BASE, "val_mcc"]; gain_all = summ.loc[best_all, "val_mcc"] - summ.loc[BASE, "val_mcc"]
gain_feat = summ.loc["Single tree + derived features", "val_mcc"] - summ.loc[BASE, "val_mcc"]
uncon = summ.loc["Unconstrained tree (default)", "val_mcc"] - summ.loc[BASE, "val_mcc"]
print(f"baseline single tree                         : val MCC {summ.loc[BASE, 'val_mcc']:.4f}")
print(f"unconstrained single tree (ceiling for 1 tree): {uncon:+.4f}")
print(f"derived features on the single tree          : {gain_feat:+.4f}")
print(f"best model that FITS the flash budget        : {gain_fit:+.4f}   ({best_fit}, ~{summ.loc[best_fit, 'exported_kb_est']:.0f} KB)")
print(f"best model overall (ignoring flash)          : {gain_all:+.4f}   ({best_all}, ~{summ.loc[best_all, 'exported_kb_est']:.0f} KB)")
print()
if gain_all < 0.003:
    print("Summary: no model gains more than 0.003 MCC over the single tree; the tree is at the ceiling of these features.")
elif gain_fit >= 0.005:
    print(f"Summary: a model that fits the flash budget gains {gain_fit:+.4f} MCC over the single tree.")
else:
    print(f"Summary: larger models gain {gain_all:+.4f} MCC, but the gain within the flash budget is {gain_fit:+.4f}.")
if gain_feat >= 0.003: print(f"Derived features alone add {gain_feat:+.4f} MCC to the single tree.")